<a href="https://colab.research.google.com/github/isaac8570/OnTime/blob/pythonmodel/ontimemodel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- 1. 라이브러리 설치 ---
!pip install firebase-admin pandas scikit-learn

import pandas as pd
import numpy as np
import firebase_admin
from firebase_admin import credentials, firestore
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import joblib

# --- 2. Firebase 연결 ---
# 🚨 Colab에 'serviceAccountKey.json' 파일 업로드 필요
try:
    cred = credentials.Certificate("serviceAccountKey.json")
    firebase_admin.initialize_app(cred)
except ValueError:
    print("Firebase 앱이 이미 초기화되었습니다.")

db = firestore.client()
print("Firebase Firestore에 연결되었습니다.")

# --- 3. 데이터 로드 및 전처리 ---
print("Firestore 'travel_logs' 컬렉션에서 데이터 로드 중...")
logs_ref = db.collection('travel_logs')
all_logs = logs_ref.stream()
logs_list = [log.to_dict() for log in all_logs]

if not logs_list:
    print("오류: 'travel_logs'에 데이터가 없습니다. 앱에서 데이터를 먼저 수집해야 합니다.")
else:
    df = pd.DataFrame(logs_list)
    print(f"총 {len(df)}개의 이동 기록을 로드했습니다.")

    # 3-1. 유효 데이터 필터링
    # 🚨 (중요) Android 앱이 이 모든 컬럼을 저장하고 있어야 합니다.
    required_cols = ['actual_eta_min', 'google_eta_min', 'user_id',
                     'weather', 'hour_of_day', 'day_of_week', 'distance_km']
    df = df.dropna(subset=required_cols)
    df = df[df['google_eta_min'] > 0]
    df = df[df['actual_eta_min'] > 0]

    # 3-2. y (타겟) 생성: ratio
    df['ratio'] = df['actual_eta_min'] / df['google_eta_min']

    # 3-3. 이상치 제거
    df = df[df['ratio'].between(0.5, 2.0)] # 0.5배 ~ 2.0배 사이의 정상 주행

    # 3-4. X (입력 피처) / y (타겟) 분리
    y = df['ratio']
    X = df[required_cols] # 'ratio'와 'actual_eta'를 제외한 모든 입력 피처 사용
    X = X.drop(['actual_eta_min'], axis=1) # actual_eta는 정답이므로 X에서 제외

    # --- 4. AI 모델 파이프라인 구축 ---

    # 4-1. 전처리 파이프라인 정의
    # 어떤 피처가 숫자형이고, 어떤 피처가 범주형(One-Hot 인코딩 필요)인지 정의
    numeric_features = ['google_eta_min', 'distance_km', 'hour_of_day']
    categorical_features = ['user_id', 'weather', 'day_of_week']

    # ColumnTransformer: 각 컬럼에 맞는 전처리를 자동으로 적용
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', numeric_features), # 숫자 피처는 그대로 통과
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features) # 범주형 피처는 원-핫 인코딩
            # 'handle_unknown='ignore'': 훈련 때 못 본 새 user_id/weather가 와도 에러내지 않음 (중요)
        ])

    # 4-2. 최종 모델 파이프라인
    # 전처리기(preprocessor)와 모델(regressor)을 하나로 묶음
    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
    ])

    # --- 5. 모델 훈련 및 저장 ---
    print("모델 훈련 시작...")
    # 전체 데이터를 사용해 최종 모델 훈련
    model_pipeline.fit(X, y)
    print("모델 훈련 완료.")

    # (선택) 훈련된 모델의 성능 평가
    print(f"모델 훈련 점수 (R^2): {model_pipeline.score(X, y):.4f}")

    # 5-1. 모델 파일로 저장
    joblib.dump(model_pipeline, 'eta_ratio_model.pkl')
    print(" 'eta_ratio_model.pkl' 파일이 성공적으로 저장되었습니다.")

FileNotFoundError: [Errno 2] No such file or directory: 'eta_model.pkl'

# 새 섹션

# 새 섹션